# Exploration: livability scoring

This is the bridge from the original ad-hoc notebook (`archive/Ideal_livable_spots.ipynb`) to the `livability` package. Everything below imports from the package — if you want to change how POIs are tagged, how fetches happen, or how scores are computed, edit `livability/*.py`, not this notebook. Changes show up immediately on the next cell run since the package is installed editable (`pip install -e ".[dev]"`).

In [ ]:
import matplotlib.pyplot as plt

from livability import (
    build_grid,
    fetch_pois,
    get_city_boundary,
    get_street_network,
    score_all,
    weighted_livability,
)
from livability.config import PLACE

PLACE

## 1. Boundary + POIs

Fetches (and disk-caches) the city boundary and the 7 categories of POIs. Re-running this cell after the first time reads from `data/.cache/` instead of hitting Overpass again.

In [ ]:
boundary = get_city_boundary(PLACE)
pois = fetch_pois(boundary)
len(pois), pois["poi_category"].value_counts()

## 2. Street network

This is the slowest step on a first run (a full city graph), and it's cached after that. If you want to iterate quickly on scoring, you can skip this cell and pass `network=None` to `score_all` — it falls back to straight-line distance.

In [ ]:
network = get_street_network(boundary.geometry.iloc[0])
network.number_of_nodes(), network.number_of_edges()

## 3. Grid + scoring (baseline)

`build_grid` / `score_all` / `weighted_livability` are the **BASELINE** — see the methodology docstring at the top of `livability/scoring.py`. This is the part meant to be improved: plug in a better distance decay function, multi-POI scoring, isochrones, etc. inside `score_category`, but keep the four signatures (`build_grid`, `score_category`, `score_all`, `weighted_livability`) stable so this notebook and `pipeline/run.py` keep working unmodified.

In [ ]:
grid = build_grid(boundary)
scores = score_all(grid, pois, network)
scores["livability"] = weighted_livability(scores)
scores.head()

## 4. Quick choropleth

Sanity-check plot, not the production visualization — that's the Next.js frontend in `web/`, which reads `data/scores.geojson` after running `python -m pipeline.run` from the repo root.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
scores.plot(column="livability", cmap="viridis", legend=True, ax=ax)
boundary.boundary.plot(ax=ax, color="black", linewidth=0.5)
ax.set_axis_off()
ax.set_title(f"{PLACE} \u2014 baseline livability score")
plt.show()

## Where to plug in improvements

- `livability/scoring.py::score_category` — currently nearest-POI distance only. Try: distance decay (e.g. exponential instead of linear min-max), counting multiple nearby POIs, or real travel-time isochrones instead of straight-line/network distance.
- `livability/scoring.py::build_grid` — currently a square grid. H3 hexagons would reduce shape distortion at city scale.
- `livability/scoring.py::weighted_livability` — currently a flat weighted sum. Try a non-linear combination, or per-cell explanations of which category drove the score.
- Once you're happy with a change, run `python -m pipeline.run` from the repo root to regenerate `data/*.geojson` and `web/public/data/*.geojson` for the frontend.